# Toffoli

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter
from io import BytesIO

import torch
from IPython.display import Image, Markdown, SVG, display
from matplotlib import pyplot as plt

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.compound_control_resource import compound_control_key
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import FixedTCall
from flagquantum.compiler.registers.control.control_predicate import ControlRegister, bind_control_registers
from flagquantum.compiler.registers.layout.int_vector import int_effect_bindings, int_state_ir
from flagquantum.compiler.registers.primitives.controlled_not import toffoli_with_controls_ir
from flagquantum.core.ir.program.register_types import quantum_bit_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg
from flagquantum.drawer.register_score import RegisterStep, layout_register_score
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic
from flagquantum.simulation.tensor_network.entrypoints import build_tensor_network, tensor_network_basis_effect
from flagquantum.simulation.unitary import get_unitary

dtype = torch.complex128

## `Toffoli`
The Toffoli gate.

This will flip the target bit if both controls are active. It can be thought of as
a reversible AND gate.

Like `TGate`, this is a common compilation target. The Clifford+Toffoli gateset is
universal.

#### References
 - [Novel constructions for the fault-tolerant Toffoli gate](https://arxiv.org/abs/1212.5069). Cody Jones. 2012. Provides a decomposition into 4 `TGate`.


In [ ]:
signature = Signature((
    Register("ctrl", quantum_bit_type(), (2,)), Register("target", quantum_bit_type()),
))

def show_score(widths, inputs, steps, wire_symbol=None):
    layout = layout_register_score(widths, inputs, steps)
    fig, _ = draw_register_score(register_score_data(layout, wire_symbol=wire_symbol))
    with BytesIO() as png:
        fig.savefig(png, format="png", bbox_inches="tight")
        display(Image(data=png.getvalue()))
    plt.close(fig)

### Example Instances

In [ ]:
toffoli = FixedTCall("toffoli")

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Toffoli", signature, {})])))

In [ ]:
ports = ("ctrl[0]", "ctrl[1]", "target")
show_score(dict.fromkeys(ports, 1), ports, (RegisterStep("Toffoli", ports, ports, "ccx"),))

### Call Graph

In [ ]:
graph = symbolic_call_graph(toffoli, {}, max_depth=1)
def title(_):
    return "Toffoli"
display(SVG(render_call_graph_svg(export_call_graph_dot(toffoli, graph.direct_calls, title=title))))
display(Markdown(format_call_graph_markdown(toffoli, graph.direct_calls, graph.leaf_counts, title=title)))
print(query_qec_gate_cost(toffoli))

## Controlled

We use `Toffoli` for specifically two control bits of the `XGate`. Requesting `Toffoli().controlled(ctrl_spec)` will give an instance of the generic `ControlledViaAnd(XGate(), ...)` bloq, which will control the `XGate` on an arbitrary boolean function supported by `CtrlSpec`. The set of control registers present on the resulting bloq will 1) the control registers needed for the provided `ctrl_spec` and 2) one additional `QBit()[2]` register for the original Toffoli control bits.

In [ ]:
def controlled_score(value):
    ports = ("extra", "ctrl[0]", "ctrl[1]", "target")
    def wire_symbol(step, port, side):
        if port is None:
            return "Text", {"text": "", "fontsize": 10}
        if step.name == "Controlled Toffoli":
            if port == "target":
                return "ModPlus", {}
            return "Circle", {"filled": bool(value) if port == "extra" else True}
        return "TextBox", {"text": port}
    print(compound_control_key(toffoli, (value,)))
    show_score(dict.fromkeys(ports, 1), ports, (
        RegisterStep("Controlled Toffoli", ports, ports),
    ), wire_symbol)

controlled_score(1)

In [ ]:
controlled_score(0)

### Complex control specs

Per above, `Toffoli.controlled()` can support arbitrary control specs.

In [ ]:
signed_six = ControlRegister(8, (6,), "twos_complement")
print(signed_six.bits())
ports = ("signed word = 6", "ctrl[0]", "ctrl[1]", "target")
show_score(dict(zip(ports, (8, 1, 1, 1))), ports, (
    RegisterStep("Controlled Toffoli", ports, ports),
))

### Bind independent control groups

The source demonstrates its controlled-adder by passing the additional unsigned eight-bit word independently of the original Toffoli ports. The native example binds the added group using the shared control-register binder and passes the original control pair and target separately. The bound kernel receives the same bit order. As in the source, integer preparation and effect surround the controlled operation.

In [ ]:
# Bind the new control word independently from the three existing ports.
word = tuple(range(8))
original = (8, 9, 10)
work = tuple(range(11, 20))
slots = tuple(range(len(work)))
controls, values = bind_control_registers((word,), (ControlRegister(8, (6,)),))
gates = toffoli_with_controls_ir(
    *original, controls=controls, values=values, workspace=work, classical_bits=slots,
)
circuit = DynamicCircuit(20, dtype=dtype)
preparation = int_state_ir(word, 6)
circuit.append(FieldOperation(20, preparation)).x(original[0]).x(original[1]).h(original[2]).s(original[2])
circuit.append(FieldOperation(20, gates))
# Use one trajectory for the larger prepared-register example.
previous_threads = torch.get_num_threads()
try:
    torch.set_num_threads(1)
    run = run_dynamic(circuit, shots=1, seed=29, strategy="trajectory")
finally:
    torch.set_num_threads(previous_threads)
state = run.final_states.reshape(256, 8, 512)
expected = torch.zeros_like(state)
expected[6, 6, 0], expected[6, 7, 0] = 1j / 2**0.5, 1 / 2**0.5
torch.testing.assert_close(state, expected, atol=2e-12, rtol=0)
# Apply the matching integer effect to the bound word.
measured = Circuit(20, inputs=state.flatten(), dtype=dtype)
plan = build_tensor_network(measured, dtype=dtype)
post = tensor_network_basis_effect(plan, int_effect_bindings(word, 6))
torch.testing.assert_close(post.reshape(8, 512), expected[6], atol=2e-12, rtol=0)

ports = ("q0", "q1", "target")
show_score({"word": 8, **dict.fromkeys(ports, 1)}, ports, (
    RegisterStep("IntState(6)", outputs=("word",)),
    RegisterStep("Controlled Toffoli", ("word", *ports), ("word", *ports)),
    RegisterStep("IntEffect(6)", ("word",)),
))
print("Controlled body emitted", dict(Counter(gate.name for gate in gates)))
print("Controlled source model", query_qec_gate_cost(compound_control_key(toffoli, values)))

### Verify the atomic action

Check the full complex matrix of the ordinary append path. The atomic gate flips only the target in the joint active sector and is self-adjoint.

In [ ]:
gates = toffoli_with_controls_ir(0, 1, 2)
circuit = Circuit(3, dtype=dtype).append(FieldOperation(3, gates))
eye = torch.eye(8, dtype=dtype)
expected = eye[:, [0, 1, 2, 3, 4, 5, 7, 6]]
torch.testing.assert_close(get_unitary(circuit), expected, atol=0, rtol=0)
circuit.append(FieldOperation(3, gates))
torch.testing.assert_close(get_unitary(circuit), eye, atol=0, rtol=0)